# Command groups: synthetic verification
Uses the repository virtual environment. No real project or external viewer is opened. GUI evidence: `Tmp/command_groups_review_20261101/after/` at the workspace root. Automated behavior and layout tests remain in `tests/test_gui_qt/test_command_editor_polish.py` and `test_command_group_save_review.py`.


In [1]:
from pathlib import Path
import subprocess
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'models/qcmodule.py').exists())
python = repo / '.venv/bin/python'
code = '''
import json
from models.qcmodule import QCModule, CommandGroup, normalize_module_commands
for count in (0, 1, 2):
    module = QCModule(name='Demo', label='Demo', commands={str(i): CommandGroup(str(i), 'viewer demo', 'shell' if i == 2 else 'direct') for i in range(1, count + 1)})
    normalize_module_commands(module)
    result = module.to_legacy_dict()
    print(json.dumps({'input_groups': count, 'code': result['code'], 'interper': result['interper'], 'commands': result.get('commands')}, ensure_ascii=False))
    assert ('commands' in result) == (count >= 2)
'''
result = subprocess.run([str(python), '-c', code], cwd=repo, text=True, capture_output=True, check=True)
print(result.stdout)


{"input_groups": 0, "code": null, "interper": "direct", "commands": null}
{"input_groups": 1, "code": "viewer demo", "interper": "direct", "commands": null}
{"input_groups": 2, "code": null, "interper": "direct", "commands": {"1": {"code": "viewer demo", "interper": "direct"}, "2": {"code": "viewer demo", "interper": "shell"}}}



In [2]:
code = '''
from models.qcmodule import QCModule
try:
    QCModule.from_legacy_dict({'name': 'Demo', 'commands': {'1': {'code': 'viewer demo', 'interper': 'unknown'}}})
except ValueError as error:
    print('Invalid mode rejected:', error)
else:
    raise AssertionError('Invalid mode was accepted')
'''
result = subprocess.run([str(python), '-c', code], cwd=repo, text=True, capture_output=True, check=True)
print(result.stdout)


Invalid mode rejected: module command interper must be 'direct' or 'shell': 'unknown'



## Limits
These cells verify serialization and invalid-mode rejection only. Full GUI, save cancellation, undo/cursor preservation, bilingual layout and launch-mode coverage are verified by the automated tests and screenshot evidence; no installed neuroimaging viewer or native Windows/macOS session is exercised.
